In [1]:
!pip install transformers torch torchvision torchaudio numpy pandas tqdm matplotlib huggingface_hub datasets evaluate scikit-learn accelerate rouge-score bert-score
!pip show transformers

Name: transformers
Version: 4.57.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: bert-score


In [2]:
import sys
import os

# Path to the folder you want to add
subfolder_path = os.path.join(os.getcwd(), "byt5_generator")

# Add it to sys.path
if subfolder_path not in sys.path:
    sys.path.append(subfolder_path)

In [3]:
import os
import argparse
import pandas as pd
import transformers

from transformers import AutoModelForSeq2SeqLM
from byt5_generator.seeding import enforce_reproducibility
from byt5_generator.dataset import prepare_datasets
from byt5_generator.train import train_seq2seq, evaluate_seq2seq

transformers.logging.set_verbosity_error()

/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

model_name = "google/byt5-base"
output_dir = f"byt5_training_results"
epochs = 3

enforce_reproducibility(42)

# ==== PREPARE DATASETS ====
train_set, val_set, test_set, tokenizer = prepare_datasets(model_name)
print(train_set[0])

# ==== MODEL ====
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ==== TRAIN MODEL ====
model, tokenizer, step_logs = train_seq2seq(
    model, train_set, val_set, tokenizer, epochs, output_dir
)

# ==== EVALUATE MODEL ====
test_logs = evaluate_seq2seq(model, tokenizer, test_set, output_dir)

[2025-10-24 10:02:17,938] - [INFO] - Train dataset has total of 45 samples
[2025-10-24 10:02:17,938] - [INFO] - Validation dataset has total of 5 samples
[2025-10-24 10:02:17,939] - [INFO] - Test dataset dataset has total of 100 samples
{'question': 'మనిషి చనిపోయాక ఏ అవయవం ఎక్కువ సమయం పనిచేస్తుంది?', 'context': 'normal body fluid volume, including blood volume, interstitial fluid volume, and intracellular fluid volume; hypovolemia and hypervolemia are imbalances. Water is necessary for all life on Earth. Humans can survive for 4 to 6 weeks without food but only for a few days without water. Profuse sweating can increase the need for electrolyte replacement. Water-electrolyte imbalance produces headache and fatigue if mild; illness if moderate, and sometimes even death if severe. For example, water intoxication (which results in hyponatremia), the process of consuming too much water too quickly, can be fatal. Deficits to body water result in volume contraction and', 'lang': 'te', 'answe